In [3]:
from __future__ import annotations

import json
import re
import sqlite3
from pathlib import Path
from typing import Any

import pandas as pd
import plotly.express as px


# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------

def natural_sort_key(value: str | Path):
    """
    Natürliche Sortierung:
    split2 kommt vor split10.
    """
    parts = re.split(r"(\d+)", str(value))

    key = []
    for part in parts:
        if part.isdigit():
            key.append((1, int(part)))
        else:
            key.append((0, part.lower()))

    return tuple(key)


def split_number(split: str) -> int:
    """
    Extrahiert aus split1, split2, ... die Zahl.
    Falls keine Zahl gefunden wird, kommt der Split nach vorne.
    """
    match = re.search(r"\d+", str(split))
    return int(match.group()) if match else -1


def normalize_metric_name(name: str) -> str:
    """
    Macht Metriknamen DataFrame-freundlicher.
    Beispiel:
        ms-ssim -> ms_ssim
    """
    return str(name).replace("-", "_")


def flatten_dict(d: dict[str, Any], prefix: str = "") -> dict[str, Any]:
    """
    Falls named_scores verschachtelte Dicts enthält, werden sie geflattet.

    Beispiel:
        {"a": {"b": 1}} -> {"a_b": 1}
    """
    out = {}

    for key, value in d.items():
        key = normalize_metric_name(key)
        new_key = f"{prefix}_{key}" if prefix else key

        if isinstance(value, dict):
            out.update(flatten_dict(value, new_key))
        else:
            out[new_key] = value

    return out


# -------------------------------------------------------------------
# Loader
# -------------------------------------------------------------------

class PromptfooMetricsLoader:
    def __init__(self, results_path: str | Path, eval_type: str):
        """
        Erwartete Struktur:

        results/
          qwen3-vl_8b-instruct/
            img-tikz/
              split1/
                promptfoo-db/
                  promptfoo.db
              split2/
                promptfoo-db/
                  promptfoo.db
              split3/
                promptfoo-db/
                  promptfoo.db

        results_path:
            Pfad zum results-Ordner.

        eval_type:
            z. B. "img-tikz".
        """
        self.results_path = Path(results_path)
        self.eval_type = eval_type

        if not self.results_path.exists():
            raise FileNotFoundError(f"results_path existiert nicht: {self.results_path}")

    def find_models(self) -> list[str]:
        """
        Findet alle Modelle, die für self.eval_type mindestens eine promptfoo.db haben.
        """
        models = []

        for model_dir in self.results_path.iterdir():
            if not model_dir.is_dir():
                continue

            db_paths = self.find_model_dbs(model_dir.name)

            if db_paths:
                models.append(model_dir.name)

        return sorted(models, key=natural_sort_key)

    def find_model_dbs(self, model: str) -> list[Path]:
        """
        Findet alle DBs eines Modells für den angegebenen eval_type.

        Unterstützt:

        results/model/img-tikz/split1/promptfoo-db/promptfoo.db
        results/model/img-tikz/split2/promptfoo-db/promptfoo.db

        Zusätzlich als Fallback:

        results/model/img-tikz/promptfoo-db/promptfoo.db
        """
        type_dir = self.results_path / model / self.eval_type

        if not type_dir.exists():
            return []

        db_paths = list(type_dir.glob("split*/promptfoo-db/promptfoo.db"))

        direct_db = type_dir / "promptfoo-db" / "promptfoo.db"
        if direct_db.exists():
            db_paths.append(direct_db)

        return sorted(db_paths, key=natural_sort_key)

    def split_name_from_db_path(self, model: str, db_path: Path) -> str:
        """
        Extrahiert split1, split2, ...
        Falls keine Split-Struktur vorhanden ist, wird "full" verwendet.
        """
        type_dir = self.results_path / model / self.eval_type
        rel = db_path.relative_to(type_dir)

        if rel.parts[0] == "promptfoo-db":
            return "full"

        return rel.parts[0]

    @staticmethod
    def parse_named_scores(value: Any) -> dict[str, Any]:
        """
        Parsed named_scores aus der promptfoo DB.
        Nicht-lesbare oder leere Werte werden ignoriert.
        """
        if value is None or value == "":
            return {}

        if isinstance(value, bytes):
            value = value.decode("utf-8")

        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            return {}

        if not isinstance(parsed, dict):
            return {}

        return flatten_dict(parsed)

    @staticmethod
    def get_eval_results_columns(conn: sqlite3.Connection) -> set[str]:
        rows = conn.execute("PRAGMA table_info(eval_results)").fetchall()
        return {row[1] for row in rows}

    def load_db(self, model: str, db_path: Path) -> pd.DataFrame:
        """
        Lädt eine einzelne promptfoo.db und annotiert die Zeilen mit:
        model, eval_type und split.
        """
        split = self.split_name_from_db_path(model, db_path)

        with sqlite3.connect(db_path) as conn:
            conn.row_factory = sqlite3.Row

            columns = self.get_eval_results_columns(conn)

            required_columns = [
                "id",
                "test_idx",
                "score",
                "success",
                "named_scores",
            ]

            missing = [col for col in required_columns if col not in columns]

            if missing:
                raise ValueError(
                    f"In {db_path} fehlen Spalten in eval_results: {missing}"
                )

            rows = conn.execute("""
                SELECT id, test_idx, score, success, named_scores
                FROM eval_results
            """).fetchall()

        data = []

        for row in rows:
            named_scores = self.parse_named_scores(row["named_scores"])

            entry = {
                "model": model,
                "eval_type": self.eval_type,
                "split": split,
                "db_path": str(db_path),
                "id": row["id"],
                "result_uid": f"{model}/{self.eval_type}/{split}/{row['id']}",
                "test_idx": row["test_idx"],
                "score": row["score"],
                "overall_score": row["score"],
                "success": row["success"],
            }

            for metric_name, metric_value in named_scores.items():
                metric_name = normalize_metric_name(metric_name)

                if metric_name in entry:
                    metric_name = f"named_{metric_name}"

                entry[metric_name] = metric_value

            data.append(entry)

        return pd.DataFrame(data)

    def load_model(self, model: str) -> pd.DataFrame:
        """
        Lädt alle Splits eines Modells und merged sie in einen DataFrame.
        """
        db_paths = self.find_model_dbs(model)

        if not db_paths:
            raise FileNotFoundError(
                f"Keine promptfoo.db gefunden für model={model}, type={self.eval_type}"
            )

        dfs = [self.load_db(model, db_path) for db_path in db_paths]

        if not dfs:
            return pd.DataFrame()

        return pd.concat(dfs, ignore_index=True)

    def load_models(self, models: list[str] | None = None) -> pd.DataFrame:
        """
        Lädt alle Modelle oder nur die angegebenen Modelle.
        """
        if models is None:
            models = self.find_models()

        if not models:
            raise ValueError(
                f"Keine Modelle für eval_type={self.eval_type} in {self.results_path} gefunden."
            )

        dfs = [self.load_model(model) for model in models]
        dfs = [df for df in dfs if not df.empty]

        if not dfs:
            raise ValueError("DBs wurden gefunden, aber keine eval_results geladen.")

        return pd.concat(dfs, ignore_index=True)

    @staticmethod
    def to_long(df: pd.DataFrame) -> pd.DataFrame:
        """
        Wandelt den breiten DataFrame in ein Long-Format für Plotly um.
        Nicht-numerische Werte werden entfernt.
        """
        meta_cols = {
            "model",
            "eval_type",
            "split",
            "db_path",
            "id",
            "result_uid",
            "test_idx",
            "score",
            "success",
        }

        metric_cols = [col for col in df.columns if col not in meta_cols]

        df_long = df.melt(
            id_vars=[col for col in df.columns if col in meta_cols],
            value_vars=metric_cols,
            var_name="metric",
            value_name="value",
        )

        def safe_numeric(value):
            if isinstance(value, (list, dict, tuple, set)):
                return None
            return value

        df_long["value"] = df_long["value"].map(safe_numeric)
        df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

        return df_long.dropna(subset=["value"])


# -------------------------------------------------------------------
# Plotting
# -------------------------------------------------------------------

def plot_metrics(
    df_long: pd.DataFrame,
    title: str,
    metrics: set[str] | None = None,
    exclude_metrics: set[str] | None = None,
):
    df_plot = df_long.copy()

    if metrics is not None:
        df_plot = df_plot[df_plot["metric"].isin(metrics)]

    if exclude_metrics is not None:
        df_plot = df_plot[~df_plot["metric"].isin(exclude_metrics)]

    if df_plot.empty:
        print(f"Keine Daten für Plot: {title}")
        return None

    fig = px.box(
        df_plot,
        x="model",
        y="value",
        color="model",
        points="all",
        facet_col="metric",
        facet_col_wrap=4,
        title=title,
        hover_data=[
            "split",
            "test_idx",
            "id",
            "success",
        ],
    )

    fig.update_xaxes(matches=None)
    fig.update_yaxes(matches=None)

    fig.update_layout(
        height=1400,
        showlegend=False,
    )

    return fig


def plot_score_metrics(df_long: pd.DataFrame):
    length_metrics = {
        "output_length",
        "reference_length",
    }

    return plot_metrics(
        df_long,
        title="Promptfoo Score Metrics by Model",
        exclude_metrics=length_metrics,
    )


def plot_length_metrics(df_long: pd.DataFrame):
    length_metrics = {
        "output_length",
        "reference_length",
    }

    return plot_metrics(
        df_long,
        title="Promptfoo Length Metrics by Model",
        metrics=length_metrics,
    )


def plot_metric_by_split(df_long: pd.DataFrame, metric: str):
    """
    Optionaler Zusatzplot:
    Zeigt eine einzelne Metrik getrennt nach Split.
    """
    df_metric = df_long[df_long["metric"] == metric].copy()

    if df_metric.empty:
        print(f"Keine Daten für Metrik: {metric}")
        return None

    fig = px.box(
        df_metric,
        x="split",
        y="value",
        color="model",
        points="all",
        facet_col="model",
        facet_col_wrap=3,
        title=f"{metric} by Split",
        hover_data=[
            "test_idx",
            "id",
            "success",
        ],
    )

    fig.update_xaxes(matches=None)
    fig.update_yaxes(matches=None)

    fig.update_layout(
        height=1000,
        showlegend=True,
    )

    return fig


# -------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------

def print_summary(df: pd.DataFrame):
    summary = (
        df.groupby(["model", "split"])
        .size()
        .reset_index(name="num_results")
    )

    summary["_model_sort"] = summary["model"].astype(str)
    summary["_split_sort"] = summary["split"].map(split_number)

    summary = (
        summary
        .sort_values(["_model_sort", "_split_sort", "split"])
        .drop(columns=["_model_sort", "_split_sort"])
    )

    print("\nGeladene Ergebnisse:")
    print(summary.to_string(index=False))

    print("\nGefundene Metriken:")

    meta_cols = {
        "model",
        "eval_type",
        "split",
        "db_path",
        "id",
        "result_uid",
        "test_idx",
        "score",
        "success",
    }

    metrics = [col for col in df.columns if col not in meta_cols]

    for metric in metrics:
        print(f"- {metric}")


# -------------------------------------------------------------------
# Usage
# -------------------------------------------------------------------

if __name__ == "__main__":
    results_path = "/usr/prakt/s0030/projects/data/results"
    eval_type = "img-tikz"

    loader = PromptfooMetricsLoader(
        results_path=results_path,
        eval_type=eval_type,
    )

    # Automatisch alle Modelle finden, die results/<model>/<eval_type>/... enthalten
    models = loader.find_models()

    print("Gefundene Modelle:")
    for model in models:
        print(f"- {model}")

    # Alle Modelle laden und alle Splits pro Modell logisch mergen
    df = loader.load_models()

    # Alternativ nur bestimmte Modelle laden:
    # df = loader.load_models([
    #     "qwen3-vl_8b-instruct",
    #     "detikzify_2_5_8b",
    # ])

    print_summary(df)

    df_long = loader.to_long(df)

    score_fig = plot_score_metrics(df_long)
    if score_fig is not None:
        score_fig.show()

    length_fig = plot_length_metrics(df_long)
    if length_fig is not None:
        length_fig.show()

    # Optional:
    # Eine einzelne Metrik nach Splits visualisieren.
    #
    # fig = plot_metric_by_split(df_long, "ms_ssim")
    # if fig is not None:
    #     fig.show()

Gefundene Modelle:
- qwen3-vl_8b-instruct

Geladene Ergebnisse:
               model    split  num_results
qwen3-vl_8b-instruct  split_1           55
qwen3-vl_8b-instruct  split_2           55
qwen3-vl_8b-instruct  split_3           55
qwen3-vl_8b-instruct  split_4           55
qwen3-vl_8b-instruct  split_5           55
qwen3-vl_8b-instruct  split_6           55
qwen3-vl_8b-instruct  split_7           55
qwen3-vl_8b-instruct  split_8           55
qwen3-vl_8b-instruct  split_9           55
qwen3-vl_8b-instruct split_10           47

Gefundene Metriken:
- overall_score
- renderable
- latex_errors
- latex_warnings
- latex_badboxes
- image_ssim
- crystalbleu
- clip_similarity
- siglip_similarity
- ted_similarity
- ted_distance
- lpips_similarity
- lpips_distance
- relative_length_error
- output_length
- reference_length
- dreamsim_similarity
- dists_distance
- ms_ssim
